# DELTA LAKE

Local instance to manage MatrizActividades

In [1]:
import os
import sys
import time
from pathlib import Path
import pandas as pd
import re
import json
import pickle
import logging
import pymongo
from pymongo.errors import ConnectionFailure
from deltalake import DeltaTable, write_deltalake
from pprint import pprint
from datetime import datetime

from eerssa.secret import Keys

logging.basicConfig(level=logging.INFO)


# Ubicación del directorio DELTA LAKE TABLE
table_path = "./test/deltalake_2025"

Success!!!


In [2]:
# --- MongoDB Connection ---
# It's better to establish the connection once and keep it open for the app's lifetime.
# We will also exit if the connection fails, as the consumer can't do its job without it.
uri = Keys.MONGO_KEY.value
client = None  # Initialize client to None
db_eerssa = None
CurrentCollection = None
ReloadCollection = None


try:
    # Add a timeout to avoid blocking indefinitely
    client = pymongo.MongoClient(uri, serverSelectionTimeoutMS=5000)
    # The ping command is cheap and does not require auth.
    client.admin.command('ping')
    db_eerssa = client.eerssa                   # Base de datos EERSSA
    CurrentCollection = db_eerssa.ot_v22        # Coleccion actual
    ReloadCollection  = db_eerssa.ot_reemplazo  # Aqui se cargan OTs repetidas
    logging.info(":::: Conexion exitosa con MongoDB ::::")
    
except ConnectionFailure as e:
    logging.error(f"\n\n ><><> Error de conexion a MongoDB: {e}")
    sys.exit(1) # Exit the script if we can't connect to MongoDB, as it's a critical dependency.


INFO:root::::: Conexion exitosa con MongoDB ::::


In [3]:
# DELTA LAKE Connection

# Verify the existence of the DELTA LAKE table
if not DeltaTable.is_deltatable(table_path):
    print(
        f"No se ha encontrado la base de datos PARQUET-DELTALAKE en la direccion:\n NO_DELTA_LAKE : {table_path}" )
else:
    print(f"Conectado a la tabla Delta Lake en: {table_path}")



Conectado a la tabla Delta Lake en: ./test/deltalake_2025


In [ ]:
# DASK

from dask.distributed import LocalCluster, as_completed
dask = LocalCluster().get_client()

INFO:distributed.scheduler:Client Client-8a8ee5f1-6672-11f0-8111-3f22e9dbfc9b requests to cancel 8550 keys
INFO:distributed.scheduler:Scheduler cancels key call_load_ot-1ef61f2d237b29607e2208d280c11ef9.  Force=False
INFO:distributed.scheduler:Client Client-8a8ee5f1-6672-11f0-8111-3f22e9dbfc9b requests to cancel 1 keys
INFO:distributed.scheduler:Scheduler cancels key call_load_ot-da9d92719a834588e6fec9ced51b7e69.  Force=False
INFO:distributed.scheduler:Scheduler cancels key GestionOt-08bc911056b570553079ee58a1b24152.  Force=False
INFO:distributed.scheduler:Client Client-8a8ee5f1-6672-11f0-8111-3f22e9dbfc9b requests to cancel 1 keys
INFO:distributed.scheduler:Scheduler cancels key call_load_ot-27e5f3166a9fa797449befb179f9ef5b.  Force=False
INFO:distributed.scheduler:Scheduler cancels key GestionOt-174ce278e6a767239fc86fa68870f839.  Force=False
INFO:distributed.scheduler:Scheduler cancels key call_load_ot-81e88024f729009ba30cfe705e3053c6.  Force=False
INFO:distributed.scheduler:Scheduler 

In [5]:
dask.dashboard_link

'http://127.0.0.1:39021/status'

### Recargar Librerias Dinámicamente


In [6]:
## RECARGAR LAS LIBRERIAS DINAMICAMENTE
from importlib import reload
from eerssa import gestionOT as OrdenTrabajo             # Convert from PDF_ot to obj_ot
from eerssa import matrizActividades as Actividades     # process ot.data["actividades"]

In [7]:
reload( OrdenTrabajo )
reload( Actividades  )

<module 'eerssa.matrizActividades' from '/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/eerssa/matrizActividades.py'>

# VERIFICACIÓN de OTs en Mongo DB y en DELTA LAKE

1. Se extrae el listado de todos los `id_ot` de los PDF existentes
2. Se verifica este listado con los documentos en `MongoDB`
3. Se verifica este listado con los documentos en `DeltaLake`

In [32]:
# 1. Listado de OTs con sus ID:

# Directorio Raiz de las OT (año) para validar

root_dir = ("/home/vlad/OneDrive/01 JEZO/01 ACTIVIDADES DIARIAS DE TRABAJO DE LAS AGENCIAS/"
            +
            "2023")



list_pdfs = []
for path in Path( root_dir ).glob("**/*.pdf"):
    list_pdfs.append( str(path) )
    list_pdfs.sort()

print(f" Se han encontrado un total de: {len(list_pdfs)} Ordenes de Trabajo" )

start_time = time.time()
start_datetime = datetime.now()
print( f"Hora de inicio: {start_datetime.strftime('%Y-%m-%d %H:%M:%S')}\n\n" )

# Helper function to call the method on the result of a future
def call_load_ot(orden_trabajo_object):
    """
    Takes the result of the first task (an OrdenTrabajo object) 
    and calls the load_ot() method on it.
    """
    return orden_trabajo_object.load_ot()

# 1. Submit the first batch of tasks
# This returns a list of futures, same as before.
futures_step1 = [dask.submit(OrdenTrabajo.GestionOt, file) for file in list_pdfs]

# 2. Submit the second batch of tasks, feeding the first futures as input
futures_step2 = [dask.submit(call_load_ot, f) for f in futures_step1]

# 3. Now, gather only the FINAL results
# This single call executes the entire graph (both GestionOt and load_ot) in parallel.
obj_lists_dask = dask.gather(futures_step2)


end_time = time.time()
elapsed_time = end_time - start_time
end_datetime = datetime.now()
print(f"\n\n   Procesados todos los {len(obj_lists_dask)} items. Tiempo transcurrido: {elapsed_time:.2f} segundos.\n   Hora Final : {end_datetime.strftime('%Y-%m-%d %H:%M:%S')}")




 Se han encontrado un total de: 4275 Ordenes de Trabajo
Hora de inicio: 2025-07-21 17:00:01




   Procesados todos los 4275 items. Tiempo transcurrido: 375.49 segundos.
   Hora Final : 2025-07-21 17:06:17


In [34]:
dask.cancel([futures_step1,futures_step2])

### Buscamos docuementos faltantes en Mongo DB

In [33]:
# 2. Obtenemos los id_OT para compararlos con la base de datos en MongoDB

# Step 1: Collect all the IDs from your local list into a new list.
all_ids = [ot.id_ot for ot in obj_lists_dask]

# Step 2: Use the "$in" operator to find all documents in the database
# that match any ID in your list. This is ONE efficient query.
existing_docs_cursor = CurrentCollection.find(
    {"id_ot": {"$in": all_ids}},
    {"id_ot": 1}  # Projection: only return the _id and id_ot fields for efficiency
)

# Step 3: Create a set of the IDs that were actually found in the database.
# Sets provide very fast lookups.
ids_in_db = {doc['id_ot'] for doc in existing_docs_cursor}

# Step 4: Find the difference between the set of all IDs and the set of IDs found in the DB.
missing_ot_ids = set(all_ids) - ids_in_db

# Print the missing IDs
for current_id in missing_ot_ids:
    print(f" [X] The ot with id: {current_id} is not in the Database ")

# Your final result is a list of the missing IDs
missing_ot = list(missing_ot_ids)


 [X] The ot with id: 0 is not in the Database 
 [X] The ot with id: 112544 is not in the Database 
 [X] The ot with id: 113025 is not in the Database 
 [X] The ot with id: 114949 is not in the Database 
 [X] The ot with id: 115080 is not in the Database 
 [X] The ot with id: 104497 is not in the Database 
 [X] The ot with id: 112595 is not in the Database 
 [X] The ot with id: 112437 is not in the Database 


In [31]:
del obj_lists_dask

#### Existen OTs en estado PDF repetidas en las carpetas

Esta es la razon de que no coincidan los numeros

In [17]:
# Buscando en MongoDB con REGEX:

csv = "/home/vlad/Documents/temp_borrar/a-reporete_del_reporte/2024-reportes/ots_query_mongo_2-24/eerssa.ot_v22 OTS del 2024.csv"

#Load file into Pandas
csv_df = pd.read_csv( csv )
csv_df.head()

regex_id = csv_df["id_ot"].tolist()
len(regex_id)

4293

In [24]:
ids_not_pdf = set(all_ids) - set(regex_id)
len(ids_not_pdf)

3

In [28]:
print(f" Total Elementos en all_ids : {len(all_ids)} Elementos unicos : {len(set(all_ids))}")

 Total Elementos en all_ids : 4316 Elementos unicos : 4296


In [30]:
from collections import Counter

pdf_contador  = Counter(all_ids)

repeated_items = {item: count for item, count in pdf_contador.items() if count > 1 }
print("\nRepeated items and their counts:")
pprint(repeated_items)


Repeated items and their counts:
{122020: 3,
 124079: 2,
 125318: 2,
 125661: 2,
 125698: 2,
 125755: 2,
 125796: 2,
 125860: 2,
 125945: 2,
 126011: 2,
 126083: 2,
 129097: 2,
 129152: 2,
 129161: 2,
 129167: 2,
 129927: 2,
 131619: 3,
 134113: 2}


## DELTA LAKE

### NUEVO DELTA LAKE Descargar todo el 2025

Para iniciar crearemos un DeltaLake de las OTs del 2025

In [ ]:
import re
# Assuming CurrentCollection is a valid pymongo.collection.Collection object
# and is already connected to your database as in consumer.py.

regex_pattern = re.compile("^2025")
query_filter = {"fecha": regex_pattern}

# Use find() to get a cursor that points to all matching documents
cursor = CurrentCollection.find(query_filter)

obj_list = []
for document in cursor:
  ot = OrdenTrabajo.GestionOt.from_dict( document )
  obj_list.append( Actividades.ConvertirOT_a_ActividadesCSV(ot))
  #print(f"Processing document with id_ot: {document.get('id_ot')}")
df = pd.concat(obj_list, ignore_index=True)


In [9]:
bkp = df.copy()

In [12]:
df.drop("uuid", axis=1, inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23437 entries, 0 to 23436
Data columns (total 23 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Item           23437 non-null  int64 
 1   Cuenta         23437 non-null  object
 2   Evento         23437 non-null  object
 3   Actividad      23390 non-null  object
 4   Alimentador    8004 non-null   object
 5   Primario       23437 non-null  object
 6   Desconexion    23437 non-null  object
 7   SIG            23437 non-null  object
 8   Tipo           20339 non-null  object
 9   Materiales     23437 non-null  object
 10  Cuadrilla      23437 non-null  object
 11  Dia            23437 non-null  object
 12  Fecha          23437 non-null  object
 13  InicioEvento   23437 non-null  object
 14  FinEvento      23435 non-null  object
 15  Duracion       23437 non-null  int64 
 16  Responsable    23437 non-null  object
 17  Colaboradores  23437 non-null  int64 
 18  HorasExtra     23437 non-n

In [14]:
# DELTA_TABLE_PATH_ON_HOST = "/var/lib/docker/volumes/delta_data/_data/my_first_delta_table"
DELTA_TABLE_PATH_ON_HOST = "./test/deltalake_2025"

In [15]:

write_deltalake(DELTA_TABLE_PATH_ON_HOST, df)

In [6]:
dt = DeltaTable("./test/deltalake_2025")
dt.version()

424

In [7]:
dt.files()

/tmp/ipykernel_31098/4072315134.py:1: DeprecationWarning: Call to deprecated method files. (Not compatible with modern delta features (e.g. shallow clones). Use `file_uris` instead.) -- Deprecated since version 1.0.0.
  dt.files()


['part-00001-17199809-0c25-400b-a2ea-cd5db9ff5a52-c000.snappy.parquet',
 'part-00001-a876d107-cc36-4a50-8469-9dbc3fc09799-c000.snappy.parquet',
 'part-00001-ffc650ec-9176-44eb-9e6a-6409832a3dd7-c000.snappy.parquet',
 'part-00001-3c8f3897-63ae-490a-b7f6-3fd9f0c37aa2-c000.snappy.parquet',
 'part-00001-4c8b0760-c246-4d7e-b334-c7b9fd6856dc-c000.snappy.parquet',
 'part-00001-dd18907b-f3bc-465d-bdda-a30b7c2e0c4f-c000.snappy.parquet',
 'part-00001-9b75687d-c228-4dd7-9f2b-f2168ec1b29d-c000.snappy.parquet',
 'part-00001-15da86d9-bf74-43e9-9a08-03531e8a9335-c000.snappy.parquet',
 'part-00001-f41201a4-0868-4f89-a57a-0e9fc66a9696-c000.snappy.parquet',
 'part-00001-70382dbb-a90c-4855-b0e8-cc633ca93c90-c000.snappy.parquet',
 'part-00001-2504a58d-b1aa-4ec4-aae0-025c1b83de8d-c000.snappy.parquet',
 'part-00001-402d43c4-ef1a-4243-90ea-096f6ad32d34-c000.snappy.parquet',
 'part-00001-21162916-ff7c-4314-974d-28d8e751cc02-c000.snappy.parquet',
 'part-00001-d9a0ba86-cf4a-4a8f-b796-e501b5dd5394-c000.snappy.pa

In [33]:
df = dt.to_pandas()

In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61960 entries, 0 to 61959
Data columns (total 23 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Item           61960 non-null  int64 
 1   Cuenta         61960 non-null  object
 2   Evento         61960 non-null  object
 3   Actividad      61813 non-null  object
 4   Alimentador    20805 non-null  object
 5   Primario       61960 non-null  object
 6   Desconexion    61960 non-null  object
 7   SIG            61960 non-null  object
 8   Tipo           53789 non-null  object
 9   Materiales     61960 non-null  object
 10  Cuadrilla      61960 non-null  object
 11  Dia            61960 non-null  object
 12  Fecha          61960 non-null  object
 13  InicioEvento   61960 non-null  object
 14  FinEvento      61958 non-null  object
 15  Duracion       61960 non-null  int64 
 16  Responsable    61960 non-null  object
 17  Colaboradores  61960 non-null  int64 
 18  HorasExtra     61960 non-n